# LangChain Hands-on 07 — Text Splitters (Chunking)
**Trainer:** Ajeetkumar

**What we will practise**
- Why split documents?
- chunk_size & chunk_overlap
- CharacterTextSplitter
- RecursiveCharacterTextSplitter (most used)
- TokenTextSplitter
- Markdown & HTML header splitters
- Code splitter (Python)
- JSON splitter

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain-text-splitters langchain-community pypdf tiktoken

# Why split?
- LLMs and embedding models have a **limit** on input size.
- Smaller chunks → more **accurate search** in RAG.
- **chunk_size** = max size of one chunk
- **chunk_overlap** = repeated text between chunks so meaning is not cut

```
|---- chunk 1 ----|
             |---- chunk 2 ----|
                          |---- chunk 3 ----|
             ^^^ overlap
```

In [ ]:
text = """LangChain is a framework for building applications with large language models.
It provides tools for prompts, chains, memory, retrieval and agents.

RAG stands for Retrieval Augmented Generation. It lets the LLM answer from your own documents.
First we load documents, then split them into chunks, then create embeddings and store them in a vector database.

Agents use an LLM to decide which tool to call. Tools can be search engines, calculators or APIs."""

In [ ]:
len(text)

## 1. CharacterTextSplitter — split on one separator

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
splitter = CharacterTextSplitter(separator="\n\n", chunk_size=200, chunk_overlap=20)
chunks = splitter.split_text(text)
len(chunks)

In [ ]:
for i, c in enumerate(chunks):
    print(i, "→", c, "\n")

## 2. ⭐ RecursiveCharacterTextSplitter — the default choice
Tries to split on `\n\n` → `\n` → space → character, so sentences stay together.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_text(text)
for i, c in enumerate(chunks):
    print(i, f"({len(c)} chars) →", c)

### Effect of chunk_size

In [ ]:
for size in [50, 100, 200, 500]:
    n = len(RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=10).split_text(text))
    print(f"chunk_size={size:>3} → {n} chunks")

### Split Documents (keeps metadata)

In [ ]:
! wget -q -O attention.pdf https://arxiv.org/pdf/1706.03762

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
pages = PyPDFLoader("attention.pdf").load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(pages)
print(len(pages), "pages →", len(chunks), "chunks")

In [ ]:
chunks[5].metadata

In [ ]:
print(chunks[5].page_content)

## 3. TokenTextSplitter — split by tokens (what LLMs actually count)

In [ ]:
from langchain_text_splitters import TokenTextSplitter

In [ ]:
splitter = TokenTextSplitter(chunk_size=30, chunk_overlap=5)
for c in splitter.split_text(text):
    print("→", c)

## 4. MarkdownHeaderTextSplitter — split by headings

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [ ]:
md_text = """# Python Course
## Module 1: Basics
Variables, data types and loops.
## Module 2: OOP
Classes, objects and inheritance.
# GenAI Course
## Module 1: LangChain
Prompts, chains and agents."""

In [ ]:
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "course"), ("##", "module")])
for d in splitter.split_text(md_text):
    print(d.metadata, "→", d.page_content)

## 5. HTMLHeaderTextSplitter

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

In [ ]:
html = "<h1>Bank FAQ</h1><h2>Cards</h2><p>Report lost cards in 3 days.</p><h2>UPI</h2><p>Never share your UPI PIN.</p>"
splitter = HTMLHeaderTextSplitter(headers_to_split_on=[("h1", "title"), ("h2", "section")])
for d in splitter.split_text(html):
    print(d.metadata, "→", d.page_content)

## 6. Code splitter — keeps functions / classes together

In [ ]:
from langchain_text_splitters import Language

In [ ]:
python_code = """
class Calculator:
    def add(self, a, b):
        return a + b

    def sub(self, a, b):
        return a - b

def greet(name):
    print(f"Hello {name}")
"""
splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, chunk_size=80, chunk_overlap=0)
for c in splitter.split_text(python_code):
    print(c, "\n-----")

In [ ]:
[l.value for l in Language][:15]

## 7. RecursiveJsonSplitter

In [ ]:
from langchain_text_splitters import RecursiveJsonSplitter

In [ ]:
data = {"company": "NovaBank", "products": {"cards": {"debit": "free", "credit": "Rs 500/yr"},
        "loans": {"home": "8.5%", "car": "9.2%", "personal": "11%"}}}
splitter = RecursiveJsonSplitter(max_chunk_size=60)
splitter.split_json(data)

# Chunking tips
| Data | Splitter | Typical size |
|---|---|---|
| normal text / PDF | RecursiveCharacterTextSplitter | 500–1000 chars, 10–20% overlap |
| markdown docs | MarkdownHeaderTextSplitter (+ recursive) | by section |
| code | `from_language(Language.PYTHON)` | by function |
| strict token limit | TokenTextSplitter | by tokens |

- Too small → loses meaning. Too big → noisy search + more cost.
- Always keep **metadata** (source, page) → for citations.

## 🧪 Practice
1. Split the PDF with chunk_size 300 and 1500 — compare number of chunks.
2. Split a markdown README by headers.
3. Split a JavaScript file using `Language.JS`.